# Public repository note

This notebook is an output-cleared code copy. It requires locally authorised data and is not runnable from the public repository alone. Green Street raw data, intermediate files, derived aggregates and outputs are not distributed.


# 05 MSOA Origin Exposure Analysis

This notebook upgrades the origin-side part of Hypothesis 1 from LAD-level exposure to MSOA-level exposure.

The reason for doing this is methodological: the residential exposure surface is defined directly from MSOA-to-MSOA commuting flows and is therefore not truncated by LAD boundaries. LADs describe the earlier data-access request, while the analytical sample is determined by the MSOAs for which retail outcome records are actually available. This produces a finer and more defensible origin-side analysis without treating individual retail properties as independent exposure observations.

## 1. Setup

In [ ]:
from pathlib import Path
import os
import json
import textwrap
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import requests
from scipy import stats
import statsmodels.api as sm
from shapely import wkb, wkt

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 160)

BASE = Path(os.environ.get("DISSERTATION_WORKSPACE", Path.cwd().resolve()))
OUT_DIR = BASE / "outputs" / "restricted_msoa_origin_exposure_analysis"
FIG_DIR = OUT_DIR / "figures"
MAP_DIR = BASE / "outputs" / "restricted_mapping_and_spatial_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "font.size": 9,
    "axes.titlesize": 11,
    "axes.labelsize": 9,
})

PALETTE = {
    "origin": "#5f9fc7",
    "origin_dark": "#2d6f9f",
    "workplace": "#d7956f",
    "green": "#8fae82",
    "gold": "#d6b36a",
    "grey": "#6f6f6f",
    "light_grey": "#eeeeea",
    "ink": "#263238",
}

figure_records = []

def add_record(figure_id, filename, theme, description, chapter_candidate):
    figure_records.append({
        "figure_id": figure_id,
        "filename": filename,
        "theme": theme,
        "description": description,
        "chapter_candidate": chapter_candidate,
    })

files = {
    "od_msoa": BASE / "ODWP01EW_MSOA.csv",
    "lad_boundaries": BASE / "Local_Authority_Districts_May_2024_Boundaries_UK_BGC_3503156029110784919.geojson",
    "openlocal_parquet": BASE / "2025-12-31-shuting-yang-greenstreet-retail.parquet",
    "workplace_lads": MAP_DIR / "workplace_destination_shock_map_table.csv",
    "selected_stations": MAP_DIR / "h1_selected_submarket_buffer_stations.csv",
    "top100_stations": BASE / "data" / "derived" / "tfl_top100_commuter_shock_stations.csv",
}

missing = [k for k, p in files.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")

print("Output directory:", OUT_DIR)

## 2. Load Boundaries and Download MSOA Boundaries if Needed

In [ ]:
london_lad = gpd.read_file(files["lad_boundaries"]).to_crs("EPSG:27700")
london_lad = london_lad[london_lad["LAD24CD"].str.startswith("E09", na=False)].copy()

msoa_local_candidates = list(BASE.glob("*MSOA*2021*Boundar*.geojson")) + list(BASE.glob("*MSOA*2021*.geojson"))
msoa_cache = OUT_DIR / "msoa_2021_boundaries_ew_bgc_v3.geojson"

def download_msoa_boundaries(cache_path):
    service_url = (
        "https://services1.arcgis.com/ESMARspQHYMw9BZ9/ArcGIS/rest/services/"
        "Middle_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V3/FeatureServer/0/query"
    )
    features = []
    offset = 0
    page_size = 2000
    while True:
        params = {
            "where": "1=1",
            "outFields": "MSOA21CD,MSOA21NM",
            "outSR": 4326,
            "f": "geojson",
            "resultOffset": offset,
            "resultRecordCount": page_size,
        }
        response = requests.get(service_url, params=params, timeout=60)
        response.raise_for_status()
        data = response.json()
        batch = data.get("features", [])
        features.extend(batch)
        if len(batch) < page_size:
            break
        offset += page_size
    geojson = {"type": "FeatureCollection", "features": features}
    cache_path.write_text(json.dumps(geojson), encoding="utf-8")
    return cache_path

if msoa_cache.exists():
    msoa_path = msoa_cache
elif msoa_local_candidates:
    msoa_path = msoa_local_candidates[0]
else:
    print("No local MSOA boundary was found. Downloading ONS MSOA 2021 BGC V3 boundaries...")
    msoa_path = download_msoa_boundaries(msoa_cache)

msoa = gpd.read_file(msoa_path).to_crs("EPSG:27700")
if "MSOA21CD" not in msoa.columns:
    raise ValueError("MSOA boundary file must contain MSOA21CD.")
if "MSOA21NM" not in msoa.columns:
    name_col = [c for c in msoa.columns if c.upper().startswith("MSOA") and c.upper().endswith("NM")]
    if name_col:
        msoa = msoa.rename(columns={name_col[0]: "MSOA21NM"})
    else:
        msoa["MSOA21NM"] = msoa["MSOA21CD"]

london_union = london_lad.geometry.union_all()
london_msoa = msoa[msoa.geometry.centroid.within(london_union)].copy()
london_msoa = london_msoa[["MSOA21CD", "MSOA21NM", "geometry"]].copy()
london_msoa.to_file(OUT_DIR / "london_msoa_2021_boundaries.geojson", driver="GeoJSON")

print("London MSOAs:", len(london_msoa))
display(london_msoa.head())

## 3. Construct MSOA-Level Residential Exposure

Station-level commuter-shock scores are first aggregated to workplace MSOAs. Where an MSOA contains more than one selected station, the aggregation uses each station's 2019 Monday-Thursday activity as a weight, so a major interchange contributes proportionately more than a small station. The resulting activity-weighted workplace shock is then combined with MSOA-to-MSOA commuting flows to construct residential exposure.

In [ ]:
london_lad_codes = set(london_lad["LAD24CD"])
london_lad_names = set(london_lad["LAD24NM"])

selected_station_table = pd.read_csv(files["selected_stations"])
selected_station_names = set(selected_station_table["clean_name"].dropna())
top100 = pd.read_csv(files["top100_stations"])
top100 = top100[top100["clean_name"].isin(selected_station_names)].dropna(subset=["geometry"]).copy()
if top100[["2019_Midweek", "cumulative_collapse_score"]].isna().any().any():
    raise ValueError("Selected stations are missing 2019 activity or commuter-shock metrics.")
top100["geometry"] = top100["geometry"].astype(str).apply(wkt.loads)
stations_gdf = gpd.GeoDataFrame(top100, geometry="geometry", crs="EPSG:4326").to_crs("EPSG:27700")
station_msoa = gpd.sjoin(
    stations_gdf[["clean_name", "2019_Midweek", "cumulative_collapse_score", "geometry"]],
    london_msoa[["MSOA21CD", "MSOA21NM", "geometry"]],
    how="left",
    predicate="within",
).drop(columns=["index_right"], errors="ignore")
missing_station_msoa = station_msoa["MSOA21CD"].isna().sum()
if missing_station_msoa:
    missing = station_msoa[station_msoa["MSOA21CD"].isna()].drop(columns=["MSOA21CD", "MSOA21NM"], errors="ignore")
    nearest = gpd.sjoin_nearest(
        missing,
        london_msoa[["MSOA21CD", "MSOA21NM", "geometry"]],
        how="left",
        distance_col="nearest_msoa_distance_m",
    ).drop(columns=["index_right"], errors="ignore")
    station_msoa = pd.concat([
        station_msoa[station_msoa["MSOA21CD"].notna()],
        nearest,
    ], ignore_index=True)

station_msoa["weighted_shock_numerator"] = (
    station_msoa["cumulative_collapse_score"]
    * station_msoa["2019_Midweek"].clip(lower=0)
)
workplace_msoa_targets = (
    pd.DataFrame(station_msoa.drop(columns="geometry"))
    .groupby(["MSOA21CD", "MSOA21NM"], as_index=False)
    .agg(
        mean_station_shock=("cumulative_collapse_score", "mean"),
        weighted_shock_numerator=("weighted_shock_numerator", "sum"),
        baseline_station_volume=("2019_Midweek", "sum"),
        max_station_shock=("cumulative_collapse_score", "max"),
        selected_station_count=("clean_name", "nunique"),
        station_names=("clean_name", lambda s: "; ".join(sorted(set(s)))),
    )
    .rename(columns={"MSOA21CD": "workplace_msoa", "MSOA21NM": "workplace_msoa_name"})
)
workplace_msoa_targets["workplace_msoa_station_shock"] = (
    workplace_msoa_targets["weighted_shock_numerator"]
    / workplace_msoa_targets["baseline_station_volume"].replace(0, np.nan)
)
workplace_msoa_targets.to_csv(OUT_DIR / "h1_selected_workplace_msoa_targets.csv", index=False)
workplace_msoa_codes = set(workplace_msoa_targets["workplace_msoa"])
print("Selected affected workplace MSOAs:", len(workplace_msoa_targets))
display(workplace_msoa_targets.head())

od_cols = [
    "Middle layer Super Output Areas code",
    "Middle layer Super Output Areas label",
    "MSOA of workplace code",
    "MSOA of workplace label",
    "Place of work indicator (4 categories) code",
    "Place of work indicator (4 categories) label",
    "Count",
]
od = pd.read_csv(files["od_msoa"], usecols=od_cols)
od = od.rename(columns={
    "Middle layer Super Output Areas code": "origin_msoa",
    "Middle layer Super Output Areas label": "origin_msoa_name",
    "MSOA of workplace code": "workplace_msoa",
    "MSOA of workplace label": "workplace_msoa_name",
    "Place of work indicator (4 categories) code": "workplace_indicator_code",
    "Place of work indicator (4 categories) label": "workplace_indicator",
    "Count": "commuters",
})
od["commuters"] = pd.to_numeric(od["commuters"], errors="coerce").fillna(0)

# Keep usual residence MSOAs in London. The residential-origin pool is
# identified directly from MSOA-to-MSOA flows to selected workplace-station
# MSOAs, rather than being inherited from the earlier LAD screening step.
od["origin_lad_name_guess"] = od["origin_msoa_name"].str.replace(r" \d{3}$", "", regex=True)
od["workplace_lad_name_guess"] = od["workplace_msoa_name"].str.replace(r" \d{3}$", "", regex=True)
od_work = od[
    od["workplace_indicator_code"].eq(3)
    & od["origin_msoa"].astype(str).str.startswith("E02", na=False)
    & od["workplace_msoa"].astype(str).str.startswith("E02", na=False)
    & od["origin_lad_name_guess"].isin(london_lad_names)
    & od["workplace_msoa"].isin(workplace_msoa_codes)
    & ~od["origin_msoa"].isin(workplace_msoa_codes)
].copy()
od_work = od_work.merge(
    workplace_msoa_targets[["workplace_msoa", "workplace_msoa_station_shock"]],
    on="workplace_msoa",
    how="left",
)
od_work["weighted_flow"] = od_work["commuters"] * od_work["workplace_msoa_station_shock"]

msoa_exposure = (
    od_work.groupby(["origin_msoa", "origin_msoa_name", "origin_lad_name_guess"], as_index=False)
    .agg(
        commuters_to_selected_workplaces=("commuters", "sum"),
        od_weighted_msoa_exposure_sum=("weighted_flow", "sum"),
        linked_workplace_msoas=("workplace_msoa", "nunique"),
        linked_workplace_lads=("workplace_lad_name_guess", "nunique"),
    )
)
msoa_exposure["msoa_exposure_per_1000"] = msoa_exposure["od_weighted_msoa_exposure_sum"] / 1000
msoa_exposure = msoa_exposure.sort_values("od_weighted_msoa_exposure_sum", ascending=False).reset_index(drop=True)
msoa_exposure.to_csv(OUT_DIR / "h1_origin_msoa_exposure.csv", index=False)

print("London origin MSOAs with positive exposure:", len(msoa_exposure))
display(msoa_exposure.head(10))

## 4. Spatially Assign OpenLocal Retail Records to MSOA

In [ ]:
target_periods = ["2019-01-01", "2023-01-01", "2024-01-01", "2025-01-01"]
openlocal_cols = [
    "period", "geocode_name", "geocode", "uarn", "occupation_state",
    "total_floor_area", "rateable_value", "geometry", "category_group"
]
ol = pd.read_parquet(
    files["openlocal_parquet"],
    columns=openlocal_cols,
    filters=[
        ("geocode", "in", list(london_lad_codes)),
        ("category_group", "==", "RETAIL"),
        ("period", "in", target_periods),
    ],
)
ol["year"] = pd.to_datetime(ol["period"]).dt.year
ol["total_floor_area"] = pd.to_numeric(ol["total_floor_area"], errors="coerce")
ol["rateable_value"] = pd.to_numeric(ol["rateable_value"], errors="coerce")
ol = ol.dropna(subset=["geometry", "uarn"]).copy()

def parse_wkb_hex(value):
    if pd.isna(value):
        return None
    try:
        return wkb.loads(bytes.fromhex(str(value)))
    except Exception:
        return None

ol["geometry"] = ol["geometry"].map(parse_wkb_hex)
ol = ol[ol["geometry"].notna()].copy()
ol_gdf = gpd.GeoDataFrame(ol, geometry="geometry", crs="EPSG:4326").to_crs("EPSG:27700")

assigned = gpd.sjoin(
    ol_gdf,
    london_msoa[["MSOA21CD", "MSOA21NM", "geometry"]],
    how="inner",
    predicate="within",
).drop(columns=["index_right"], errors="ignore")
assigned = assigned[~assigned["MSOA21CD"].isin(workplace_msoa_codes)].copy()
assigned = assigned.merge(
    msoa_exposure[["origin_msoa", "msoa_exposure_per_1000", "od_weighted_msoa_exposure_sum", "commuters_to_selected_workplaces"]],
    left_on="MSOA21CD",
    right_on="origin_msoa",
    how="left",
)
assigned["msoa_exposure_per_1000"] = assigned["msoa_exposure_per_1000"].fillna(0)
assigned["od_weighted_msoa_exposure_sum"] = assigned["od_weighted_msoa_exposure_sum"].fillna(0)
assigned["commuters_to_selected_workplaces"] = assigned["commuters_to_selected_workplaces"].fillna(0)

print("OpenLocal retail records assigned to MSOA:", len(assigned))
print("Unique UARN:", assigned["uarn"].nunique())
print("MSOAs with OpenLocal records:", assigned["MSOA21CD"].nunique())
assigned.drop(columns="geometry").to_csv(OUT_DIR / "openlocal_retail_records_assigned_to_msoa_restricted.csv", index=False)
display(assigned.head())

## 5. Build MSOA-Year Retail Indicators

In [ ]:
msoa_year = (
    assigned.groupby(["MSOA21CD", "MSOA21NM", "year"], as_index=False)
    .agg(
        retail_records=("uarn", "size"),
        retail_units=("uarn", "nunique"),
        total_rateable_value=("rateable_value", "sum"),
        median_rateable_value=("rateable_value", "median"),
        total_floor_area=("total_floor_area", "sum"),
        median_floor_area=("total_floor_area", "median"),
        vacant_records=("occupation_state", lambda s: s.astype(str).str.upper().eq("VACANT").sum()),
        msoa_exposure_per_1000=("msoa_exposure_per_1000", "first"),
        od_weighted_msoa_exposure_sum=("od_weighted_msoa_exposure_sum", "first"),
        commuters_to_selected_workplaces=("commuters_to_selected_workplaces", "first"),
    )
)
msoa_year["vacancy_proxy"] = msoa_year["vacant_records"] / msoa_year["retail_records"].replace(0, np.nan)
msoa_year.to_csv(OUT_DIR / "openlocal_msoa_year_indicators.csv", index=False)

base = msoa_year[msoa_year["year"].eq(2019)].drop_duplicates("MSOA21CD")
post = msoa_year[msoa_year["year"].isin([2023, 2024, 2025])]
post_mean = (
    post.groupby(["MSOA21CD", "MSOA21NM"], as_index=False)
    .agg(
        retail_units_post=("retail_units", "mean"),
        total_rateable_value_post=("total_rateable_value", "mean"),
        median_rateable_value_post=("median_rateable_value", "mean"),
        total_floor_area_post=("total_floor_area", "mean"),
        median_floor_area_post=("median_floor_area", "mean"),
        vacancy_proxy_post=("vacancy_proxy", "mean"),
    )
)
msoa_change = (
    base[[
        "MSOA21CD", "MSOA21NM", "retail_units", "total_rateable_value", "median_rateable_value",
        "total_floor_area", "median_floor_area", "vacancy_proxy", "msoa_exposure_per_1000",
        "od_weighted_msoa_exposure_sum", "commuters_to_selected_workplaces"
    ]]
    .merge(post_mean, on=["MSOA21CD", "MSOA21NM"], how="inner")
)
for metric in ["retail_units", "total_rateable_value", "median_rateable_value", "total_floor_area", "median_floor_area"]:
    msoa_change[f"{metric}_pct_change"] = (
        msoa_change[f"{metric}_post"] - msoa_change[metric]
    ) / msoa_change[metric].replace(0, np.nan)
msoa_change["vacancy_proxy_change"] = msoa_change["vacancy_proxy_post"] - msoa_change["vacancy_proxy"]
msoa_change.to_csv(OUT_DIR / "openlocal_msoa_2019_to_post_mean_change_indicators.csv", index=False)

yearly_change = base[[
    "MSOA21CD", "MSOA21NM", "retail_units", "total_rateable_value", "median_rateable_value",
    "total_floor_area", "median_floor_area", "vacancy_proxy", "msoa_exposure_per_1000",
    "od_weighted_msoa_exposure_sum", "commuters_to_selected_workplaces"
]].rename(columns={
    "retail_units": "retail_units_2019",
    "total_rateable_value": "total_rateable_value_2019",
    "median_rateable_value": "median_rateable_value_2019",
    "total_floor_area": "total_floor_area_2019",
    "median_floor_area": "median_floor_area_2019",
    "vacancy_proxy": "vacancy_proxy_2019",
}).merge(post, on=["MSOA21CD", "MSOA21NM"], how="inner", suffixes=("", "_year"))
for metric in ["retail_units", "total_rateable_value", "median_rateable_value", "total_floor_area", "median_floor_area"]:
    yearly_change[f"{metric}_pct_change_2019"] = (
        yearly_change[metric] - yearly_change[f"{metric}_2019"]
    ) / yearly_change[f"{metric}_2019"].replace(0, np.nan)
yearly_change["vacancy_proxy_change_2019"] = yearly_change["vacancy_proxy"] - yearly_change["vacancy_proxy_2019"]
yearly_change.to_csv(OUT_DIR / "openlocal_msoa_yearly_change_indicators.csv", index=False)

print("MSOA change sample:", msoa_change.shape)
display(msoa_change.head())

### 5.1 Distinguish Exposure Coverage from the Testable Analysis Sample

In [ ]:
exposure_availability = london_msoa[["MSOA21CD", "MSOA21NM", "geometry"]].merge(
    msoa_exposure[["origin_msoa", "msoa_exposure_per_1000"]],
    left_on="MSOA21CD",
    right_on="origin_msoa",
    how="left",
)
exposure_availability["msoa_exposure_per_1000"] = exposure_availability["msoa_exposure_per_1000"].fillna(0)
exposure_availability["has_positive_ons_exposure"] = exposure_availability["msoa_exposure_per_1000"].gt(0)
exposure_availability["is_workplace_station_msoa"] = exposure_availability["MSOA21CD"].isin(workplace_msoa_codes)
exposure_availability["has_supplied_retail_records"] = exposure_availability["MSOA21CD"].isin(
    set(msoa_year["MSOA21CD"])
)
exposure_availability["has_complete_2019_post_outcomes"] = exposure_availability["MSOA21CD"].isin(
    set(msoa_change["MSOA21CD"])
)
core_model_codes = set(
    msoa_change.loc[
        msoa_change["msoa_exposure_per_1000"].gt(0)
        & msoa_change["retail_units"].ge(20)
        & msoa_change["retail_units_pct_change"].notna(),
        "MSOA21CD",
    ]
)
exposure_availability["included_in_core_model"] = exposure_availability["MSOA21CD"].isin(core_model_codes)
exposure_availability.drop(columns="geometry").to_csv(
    OUT_DIR / "h1_origin_msoa_analysis_availability.csv", index=False
)

analysis_eligible = exposure_availability[
    exposure_availability["has_positive_ons_exposure"]
    & ~exposure_availability["is_workplace_station_msoa"]
    & exposure_availability["has_complete_2019_post_outcomes"]
]
exposed_without_outcomes = exposure_availability[
    exposure_availability["has_positive_ons_exposure"]
    & ~exposure_availability["is_workplace_station_msoa"]
    & ~exposure_availability["has_complete_2019_post_outcomes"]
]

print("Positive-exposure residential MSOAs:", int((exposure_availability["has_positive_ons_exposure"] & ~exposure_availability["is_workplace_station_msoa"]).sum()))
print("With complete 2019/post outcomes:", len(analysis_eligible))
print("Eligible for the core retail-unit model:", len(core_model_codes))

## 6. MSOA-Level Scatterplots and Models

In [ ]:
def scatter_fit(ax, df, x, y, color, xlabel, ylabel, title):
    clean = df[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    clean = clean[clean[x].gt(0)].copy()
    ax.scatter(clean[x], clean[y], s=28, color=color, alpha=0.68, edgecolor="white", linewidth=0.25)
    if len(clean) >= 10 and clean[x].nunique() > 1:
        coef, intercept = np.polyfit(clean[x], clean[y], 1)
        xs = np.linspace(clean[x].min(), clean[x].max(), 120)
        ax.plot(xs, intercept + coef * xs, color=PALETTE["ink"], linewidth=1.0)
        rho, pval = stats.spearmanr(clean[x], clean[y])
        ax.text(
            0.02, 0.98,
            f"MSOAs={len(clean)}; Spearman rho={rho:.2f}; p={pval:.3g}",
            transform=ax.transAxes,
            va="top",
            ha="left",
            fontsize=8,
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.76, "pad": 2.5},
        )
    ax.axhline(0, color="#9a9a9a", linewidth=0.7)
    ax.grid(alpha=0.22)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title, loc="left")

fig, axes = plt.subplots(1, 3, figsize=(14.2, 4.6))
outcomes = [
    ("total_rateable_value_pct_change", "Total rateable value", "Pct change, 2023-2025 mean vs 2019"),
    ("retail_units_pct_change", "Retail unit count", "Pct change, 2023-2025 mean vs 2019"),
    ("total_floor_area_pct_change", "Retail floor area", "Pct change, 2023-2025 mean vs 2019"),
]
for ax, (outcome, title, ylabel) in zip(axes, outcomes):
    scatter_fit(
        ax,
        msoa_change,
        "msoa_exposure_per_1000",
        outcome,
        PALETTE["origin"],
        "MSOA residential exposure (weighted flow / 1,000)",
        ylabel,
        title,
    )
fig.suptitle("H1 Origin-Side MSOA Continuous Exposure Test", x=0.02, ha="left", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.93])
p = FIG_DIR / "fig_02_msoa_origin_continuous_scatter.png"
fig.savefig(p, bbox_inches="tight")
plt.show()
add_record("Figure 5.x", p.name, "H1 origin MSOA", "MSOA-level relationship between residential exposure and OpenLocal retail aggregate change.", "Chapter 5 Results")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14.8, 8.2), sharex=True)
yearly_outcomes = [
    ("total_rateable_value_pct_change_2019", "Total rateable value", PALETTE["origin"]),
    ("retail_units_pct_change_2019", "Retail unit count", PALETTE["origin_dark"]),
]
for row, (outcome, row_title, color) in enumerate(yearly_outcomes):
    all_clean = yearly_change[["msoa_exposure_per_1000", outcome]].replace([np.inf, -np.inf], np.nan).dropna()
    all_clean = all_clean[all_clean["msoa_exposure_per_1000"].gt(0)]
    y_min, y_max = all_clean[outcome].quantile([0.02, 0.98]) if len(all_clean) else (-1, 1)
    pad = (y_max - y_min) * 0.18 if y_max > y_min else 1
    for col, year in enumerate([2023, 2024, 2025]):
        ax = axes[row, col]
        df_year = yearly_change[yearly_change["year"].eq(year)]
        scatter_fit(
            ax,
            df_year,
            "msoa_exposure_per_1000",
            outcome,
            color,
            "MSOA residential exposure (weighted flow / 1,000)" if row == 1 else "",
            f"{row_title} pct change\nyear minus 2019" if col == 0 else "",
            f"{row_title}: {year} vs 2019",
        )
        ax.set_ylim(y_min - pad, y_max + pad)
fig.suptitle("H1 Origin-Side MSOA Yearly Exposure Test", x=0.02, ha="left", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.95])
p = FIG_DIR / "fig_03_msoa_origin_yearly_scatter.png"
fig.savefig(p, bbox_inches="tight")
plt.show()
add_record("Figure 5.x", p.name, "H1 origin MSOA", "Yearly MSOA-level relationship between residential exposure and OpenLocal retail outcomes.", "Chapter 5 Results")

In [ ]:
def run_msoa_ols(df, outcome, predictor="msoa_exposure_per_1000", controls=None, label=None, min_units=20):
    controls = controls or []
    cols = [outcome, predictor, "retail_units"] + controls
    clean = df[cols].replace([np.inf, -np.inf], np.nan).dropna()
    clean = clean[clean[predictor].gt(0) & clean["retail_units"].ge(min_units)].copy()
    if len(clean) < 20 or clean[predictor].nunique() < 2:
        return None
    X = sm.add_constant(clean[[predictor] + controls].astype(float))
    y = clean[outcome].astype(float)
    fit = sm.WLS(y, X, weights=np.sqrt(clean["retail_units"].astype(float))).fit(cov_type="HC3")
    rho, pval = stats.spearmanr(clean[predictor], clean[outcome])
    return {
        "model": label or outcome,
        "sample": "OpenLocal residential-origin MSOAs",
        "outcome": outcome,
        "predictor": predictor,
        "n_obs": int(fit.nobs),
        "beta": fit.params[predictor],
        "robust_se": fit.bse[predictor],
        "t_value": fit.tvalues[predictor],
        "p_value": fit.pvalues[predictor],
        "r_squared": fit.rsquared,
        "spearman_r": rho,
        "spearman_p": pval,
        "controls": ", ".join(controls) if controls else "none",
    }

msoa_models = pd.DataFrame([
    run_msoa_ols(msoa_change, "total_rateable_value_pct_change", label="M1 total rateable value"),
    run_msoa_ols(msoa_change, "retail_units_pct_change", label="M2 retail unit count"),
    run_msoa_ols(msoa_change, "total_floor_area_pct_change", label="M3 total floor area"),
    run_msoa_ols(msoa_change, "median_rateable_value_pct_change", label="M4 median rateable value"),
    run_msoa_ols(msoa_change, "vacancy_proxy_change", label="M5 vacancy proxy"),
]).dropna(how="all")
msoa_models.to_csv(OUT_DIR / "h1_origin_msoa_models.csv", index=False)
display(msoa_models)

In [ ]:
yearly_model_specs = [
    ("Total rateable value", "total_rateable_value_pct_change_2019"),
    ("Retail unit count", "retail_units_pct_change_2019"),
    ("Retail floor area", "total_floor_area_pct_change_2019"),
]
yearly_model_rows = []
for year in [2023, 2024, 2025]:
    df_year = yearly_change[yearly_change["year"].eq(year)].copy()
    for label, outcome in yearly_model_specs:
        result = run_msoa_ols(df_year, outcome, label=f"{label} {year}")
        if result is not None:
            result["year"] = year
            result["outcome_label"] = label
            yearly_model_rows.append(result)
msoa_yearly_models = pd.DataFrame(yearly_model_rows)
msoa_yearly_models.to_csv(OUT_DIR / "h1_origin_msoa_yearly_models.csv", index=False)

coef_plot = msoa_yearly_models.copy()
coef_plot["ci_low"] = coef_plot["beta"] - 1.96 * coef_plot["robust_se"]
coef_plot["ci_high"] = coef_plot["beta"] + 1.96 * coef_plot["robust_se"]

fig, axes = plt.subplots(1, 3, figsize=(12.8, 3.9), sharey=False)
for ax, (label, _) in zip(axes, yearly_model_specs):
    sub = coef_plot[coef_plot["outcome_label"].eq(label)].sort_values("year")
    x = np.arange(len(sub))
    ax.axhline(0, color="#8f8f8f", linewidth=0.8)
    ax.vlines(x, sub["ci_low"], sub["ci_high"], color="#9bbbd0", linewidth=3, alpha=0.85)
    ax.scatter(x, sub["beta"], color=PALETTE["origin_dark"], s=46, zorder=3)
    ax.set_xticks(x)
    ax.set_xticklabels(sub["year"].astype(str))
    ax.set_title(label, loc="left")
    ax.set_xlabel("Year vs 2019")
    ax.set_ylabel("Exposure coefficient")
    ax.grid(axis="y", alpha=0.22)
    for xi, (_, row) in zip(x, sub.iterrows()):
        ax.text(xi, row["ci_high"], f"p={row['p_value']:.2g}", ha="center", va="bottom", fontsize=7.5)
fig.suptitle("MSOA Origin-Side Yearly Exposure Coefficients", x=0.02, ha="left", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.90])
p = FIG_DIR / "fig_05_msoa_origin_yearly_coefficients.png"
fig.savefig(p, bbox_inches="tight")
plt.show()
add_record("Figure 5.x", p.name, "H1 origin MSOA", "Yearly MSOA-level exposure coefficients for OpenLocal retail outcomes.", "Chapter 5 Results")

display(msoa_yearly_models)

## 7. Save Model Table and Figure Manifest

In [ ]:
display_cols = ["model", "sample", "n_obs", "beta", "robust_se", "t_value", "p_value", "r_squared", "spearman_r", "spearman_p", "controls"]
model_display = msoa_models[display_cols].copy()
for col in ["beta", "robust_se", "t_value", "p_value", "r_squared", "spearman_r", "spearman_p"]:
    model_display[col] = model_display[col].map(lambda v: f"{v:.3g}" if pd.notna(v) else "")

fig_height = max(2.4, 0.42 * (len(model_display) + 2))
fig, ax = plt.subplots(figsize=(13.2, fig_height))
ax.axis("off")
ax.set_title("H1 Origin-Side MSOA Continuous Model Results", loc="left", fontsize=12, pad=10)
table = ax.table(cellText=model_display.values, colLabels=model_display.columns, loc="center", cellLoc="left", colLoc="left")
table.auto_set_font_size(False)
table.set_fontsize(7.3)
table.scale(1, 1.25)
for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor("#d0d0d0")
    cell.set_linewidth(0.35)
    if row == 0:
        cell.set_facecolor("#e9edf2")
        cell.set_text_props(weight="bold")
    elif row % 2 == 0:
        cell.set_facecolor("#f8f9fb")
fig.tight_layout()
p = FIG_DIR / "fig_04_msoa_origin_model_results_table.png"
fig.savefig(p, bbox_inches="tight")
plt.show()
add_record("Table 5.x", p.name, "H1 origin MSOA", "MSOA-level origin-side continuous exposure model results.", "Chapter 5 Results")

figure_manifest = pd.DataFrame(figure_records)
figure_manifest.to_csv(OUT_DIR / "figure_manifest.csv", index=False)
display(figure_manifest)
print("Saved outputs to:", OUT_DIR)
print("Saved figures to:", FIG_DIR)

## 9. Reading Guide

This notebook should be treated as an upgraded origin-side analysis for Hypothesis 1.

- This notebook is the preferred H1 origin-side analysis: Top-100 affected stations are filtered to selected workplace stations, mapped to workplace MSOAs, and then linked to residential MSOAs through the ONS MSOA OD table.
- The first exposure map is the ONS-derived exposure universe. The second map is the testable analysis sample: blue MSOAs also have complete OpenLocal outcome data, while hatched MSOAs have calculated exposure but insufficient supplied retail outcomes.
- The previous LAD-level result remains useful as a screening and robustness check because it matches the earlier data-access geography discussed with the supervisor.
- Individual OpenLocal property records are not used as independent exposure observations. They are first assigned to MSOAs and aggregated to MSOA-year indicators, which avoids inflating the sample size artificially.